# Ch 10. 임베딩과 벡터 검색 기초

이 노트북은 [AI Assistant Engineering - Part 3, Ch 10](https://desty.github.io/study-ai-assistant-engineering/part3/10-embedding/) 의 실습용 보조 자료입니다.

## 주제
- **임베딩(embedding)** — 문장을 고차원 숫자 벡터로 바꾸는 일과 왜 그게 유용한가
- **코사인 유사도**로 "의미가 비슷하다" 를 측정
- **벡터 DB** (Chroma · Qdrant · Pinecone) 가 하는 일
- **ANN**(Approximate Nearest Neighbor) 의 존재 이유
- 차원 불일치 · 언어 혼용 · 정규화 같은 **현실의 함정**

In [ ]:
!pip install -q chromadb openai

In [ ]:
import os
from getpass import getpass

for k in ['OPENAI_API_KEY']:
    if not os.getenv(k):
        os.environ[k] = getpass(f'{k}: ')

## 1. 개념 — 문장을 숫자로

사람은 "**강아지**"와 "**고양이**"가 비슷하고, "**강아지**"와 "**피자**"가 멀다는 걸 직관적으로 압니다.
컴퓨터는? **문자 자체로는** 강아지·고양이·피자가 모두 "서로 다른 문자열".

**임베딩**은 이 직관을 숫자로 바꾸는 기법:
- "강아지" → [0.12, -0.03, 0.87, ..., 0.41]  (1536개 수)
- "고양이" → [0.14, -0.01, 0.82, ..., 0.39]  (강아지와 거의 비슷한 방향)
- "피자"   → [-0.40, 0.78, -0.12, ..., 0.05] (완전히 다른 방향)

비슷한 의미는 **가까운 벡터**로. 그게 전부입니다.

## 4. 최소 예제 — 문장 3개의 유사도

In [ ]:
from openai import OpenAI
import numpy as np

client = OpenAI()

sentences = ["강아지가 짖는다", "개가 멍멍 운다", "피자가 맛있다"]

res = client.embeddings.create(
    model="text-embedding-3-small",
    input=sentences,
)
vecs = np.array([d.embedding for d in res.data])

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"강아지-개:   {cosine(vecs[0], vecs[1]):.3f}")   # 약 0.85~0.95 예상
print(f"강아지-피자: {cosine(vecs[0], vecs[2]):.3f}")   # 약 0.20~0.40 예상

"강아지-개" 는 0.9 근처 · "강아지-피자" 는 0.3 근처 로 나와야 **임베딩 모델이 잘 작동하는 것**.
이게 0.5 이상이면 모델이 너무 뭉툭하거나 한국어 성능이 약하다는 신호.

## 5. 실전 튜토리얼

### 5.2 코사인 유사도 · 정규화

In [ ]:
def cosine_normalized(a, b):
    return np.dot(a, b)  # 정규화된 벡터라면 dot product = cosine similarity

# OpenAI/Voyage 임베딩은 대부분 L2 normalized (length=1) 로 옴
# → 성능 최적화를 위해 미리 정규화해 저장, 검색 시 dot product 만

# 검증
normalized_vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
print(f"\n정규화 후 dot product: {np.dot(normalized_vecs[0], normalized_vecs[1]):.3f}")
print(f"원래 cosine: {cosine(vecs[0], vecs[1]):.3f}")

### 5.3 벡터 DB — Chroma 실습

In [ ]:
import chromadb

chroma = chromadb.PersistentClient(path="/tmp/chroma_db_ch10")

def embed(texts: list[str]) -> list[list[float]]:
    res = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return [d.embedding for d in res.data]

# 컬렉션 생성 (테이블 느낌)
col = chroma.get_or_create_collection(name="faq_kb")

docs = [
    "환불은 구매 후 7일 이내 신청 가능. 팀장 승인 필요.",
    "배송은 주문일 기준 영업일 2~3일 소요.",
    "포인트는 구매 금액의 1% 적립, 3개월 이내 사용.",
]
col.add(
    ids=[f"doc-{i}" for i in range(len(docs))],
    documents=docs,
    embeddings=embed(docs),
    metadatas=[{"source": "policy.md"} for _ in docs],
)

print("문서 저장 완료")

In [ ]:
# 검색
query = "돈 돌려받으려면?"
q_emb = embed([query])[0]
result = col.query(query_embeddings=[q_emb], n_results=2)

for doc, meta, dist in zip(result["documents"][0], result["metadatas"][0], result["distances"][0]):
    print(f"distance={dist:.3f}  [{meta['source']}]  {doc}")

### 5.4 ANN — 왜 완전 탐색이 아닌가

100만 문서에서 top-10 찾으려면 **100만 번 코사인** 계산. 너무 느림.

**ANN (Approximate Nearest Neighbor)** 는 **근사 답** 을 빠르게 반환:
- **HNSW** (Hierarchical Navigable Small World) — 계층적 그래프 탐색. Chroma·Qdrant 기본
- **IVF** (Inverted File) — 클러스터 먼저 좁히기
- **PQ** (Product Quantization) — 벡터 압축으로 메모리 절약

정확도 99% 라면 **100배 빠르게** 얻음. 이 시점에선 "ANN 이 있다" 만 알아도 OK — 벡터 DB가 알아서 씀.

### 5.5 쿼리 vs 문서 임베딩 — Asymmetric

같은 모델로 쿼리와 문서를 임베딩할 때 보통 OK. 하지만 **긴 문서 vs 짧은 쿼리** 에서는 최신 모델들이 **role 지시** 를 지원합니다.

**Voyage** 예:
```python
# 문서 임베딩 (저장용)
client.embed(texts=docs, input_type="document")

# 쿼리 임베딩 (검색용)
client.embed(texts=[query], input_type="query")
```

이걸 **symmetric vs asymmetric** 임베딩이라고 부름. 정확도 차이가 크면 쿼리/문서별로 다르게.

## 6. 자주 깨지는 포인트

### 실수 1. 차원 불일치
OpenAI small(1536) 로 임베딩한 컬렉션에 Voyage(1024) 임베딩을 추가하면 폭발.

**대응**: 컬렉션 메타데이터에 `embedding_model` 기록. 모델 변경 시 **전체 재임베딩**.

### 실수 2. 정규화 여부 혼선
어떤 모델은 L2 normalized (length=1) 로 반환, 어떤 건 아님.

**대응**: 임베딩 직후 `vec / np.linalg.norm(vec)` 로 강제 정규화.

### 실수 3. 언어 혼용
한국어 문서 + 영어 쿼리 (또는 반대) 는 **다국어 모델** 아니면 품질 급락.

**대응**: Voyage · BGE-M3 · OpenAI `text-embedding-3-large` 중 다국어 벤치 확인.

### 실수 4. 짧은 쿼리에 과대해석
"환불" 한 단어 쿼리는 임베딩의 분산이 커서 노이즈.

**대응**: (a) HyDE (Ch 13) 로 가상 답변을 임베딩, (b) 키워드 하이브리드 검색 (Ch 12), (c) 쿼리 확장 프롬프트.

### 실수 5. 메타데이터 없이 저장
나중에 "이 답이 어느 문서/어느 섹션에서 왔냐" 를 모름.

**대응**: **저장 시점**에 `source`, `page`, `section`, `updated_at`, `owner` 같은 메타 항상 함께.

## 8. 확인 문제

1. 위 `similarity.py` 를 한국어 · 영어 · 동의어·반의어 쌍으로 10개 돌려 유사도 분포 표 만들기
2. Chroma 예제에 자기 FAQ 5~10건 넣고 다양한 쿼리로 top-k 품질 관찰
3. 같은 문서에 **OpenAI small vs Voyage-3** 를 둘 다 돌려 상위 3개 결과가 얼마나 다른지 비교
4. 일부러 **정규화 안 한 벡터**로 cosine similarity 계산해 값이 어떻게 튀는지 확인
5. 한국어 질문에 영어 문서 섞어 놓고 top-k 품질 측정

**다음 챕터** → [Ch 11. RAG 파이프라인 전체 흐름](https://desty.github.io/study-ai-assistant-engineering/part3/11-pipeline/)

임베딩 + 벡터 DB 는 재료. 다음은 **문서 수집부터 citation까지 end-to-end** 흐름.